In [ ]:
%reset -f

In [5]:
import os
import numpy as np
import pandas as pd
import xarray as xr
import warnings as ws
ws.filterwarnings("ignore")

In [2]:
os.chdir(r"G:\Theses\Dossier_ma_These\Data\Simulations_ORCHIDEE")

In [15]:
### Sechiba folder
#nomenclature:
"""
    EXP-CompRain.CompGW = R_avgSWCC_avg
    EXP-CompRain.FfluxGW = R_avgSWCC_var
    EXP-RainFflux.CompGW = R_varSWCC_avg
    EXP-RainFflux.GWFflux = R_varSWCC_var
"""
name_simulation = "EXP-CompRain.CompGW"

output_nc = xr.open_mfdataset("EXP-CompRain.CompGW\SRF\Output\MO\EXP-CompRain.CompGW_*_sechiba_history.nc")
output_4dim_nc = xr.open_mfdataset("EXP-CompRain.CompGW\SRF\Output\MO\EXP-CompRain.CompGW_*_sechiba_history_4dim.nc")


In [21]:
#Conversion
def kgC_m2_s_to_gC_m2_day(kgC_m2_s):
    """
    Convert Gross Primary Production (GPP) from kgC/m²/s to gC/m²/day.
    Parameters:
    kgC_m2_s (float or np.array): carbon flux in kgC/m²/s.
    Returns:
    float or np.array: carbon in gC/m²/day.
    """
    # Conversion factors
    kg_to_g = 1000  # 1 kg = 1000 g
    seconds_to_day = 86400  # 1 day = 86400 seconds
    # Convert C fluxes to gC/m²/day
    gC_m2_day = kgC_m2_s * kg_to_g * seconds_to_day

    return gC_m2_day




In [22]:
def w_m2_to_mj_m2_day(w_m2):
    """
    Convert Latent Heat Flux (LE) from W/m² to MJ/m²/day.

    Parameters:
    le_w_per_m2 (float or np.array): Latent Heat Flux in W/m².

    Returns:
    - 1 Megajoule (MJ) = 1,000,000 Joules (J), so we divide by `1,000,000` to convert Joules to Megajoules.
    - There are `86400` seconds in a day, so to get the total energy per day, multiply the value by `86400`
    float or np.array: Latent Heat Flux in MJ/m²/day.
    """
    # Conversion factor from W/m² to MJ/m²/day
    conversion_factor = 0.0864

    # Convert LE to MJ/m²/day
    mj_m2_day = w_m2 * conversion_factor

    return mj_m2_day

In [35]:
# Function to process a single simulation
def process_simulation(name_simulation, prefix, variables, file_pattern, solay_value=None):
    # Open the dataset
    output_nc = xr.open_mfdataset(f"{name_simulation}/SRF/Output/MO/{file_pattern}")
    
    # If 4-dimensional data is required (e.g., moistc with solay)
    if 'moistc' in variables and solay_value is not None:
        print("yes")
        output_4dim_nc = xr.open_mfdataset(f"{name_simulation}/SRF/Output/MO/{name_simulation}_*_sechiba_history_4dim.nc")
        moistc_data = output_4dim_nc.moistc.sel(solay=solay_value, method="nearest").to_dataframe().rename(columns={"moistc": f"{prefix}_moistc"})
        moistc_data.head()
    else:
        moistc_data = pd.DataFrame()  # Empty if not applicable
    print("yes2")
    moistc_data.head()
    # Extract the required variables and add prefix
    data = output_nc[variables].to_dataframe()
    data = data.rename(columns={var: f"{var}_{prefix}" for var in variables})
    print("yes3")
    # Apply GPP conversion (kgC/m²/s to gC/m²/day) to any column containing 'gpp'
    for col in data.columns:
        if "gpp" in col.lower():  # Check if the variable name contains 'gpp'
            data[col] = kgC_m2_s_to_gC_m2_day(data[col])
        elif "fluxsens" in col.lower() or "fluxlat" in col.lower():  # Check if the variable name contains 'fluxsens' or 'fluxlat'
            data[col] = w_m2_to_mj_m2_day(data[col])
    
    # Combine with moistc data if available
    if not moistc_data.empty:
        
        data = data.join(moistc_data, how="left")
    
    # return data

# Define simulation details
simulations = {
    "EXP-CompRain.CompGW": "R_avgSWCC_avg",
    "EXP-CompRain.FfluxGW": "R_avgSWCC_var",
    "EXP-RainFflux.CompGW": "R_varSWCC_avg",
    "EXP-RainFflux.GWFflux": "R_varSWCC_var",
}

variables = ["moistc", "rain", "gppTree", "gppCrop", "fluxlat", "fluxsens"]

# Process each simulation and combine into one dataframe
df_list = []
for name_simulation, prefix in simulations.items():
    df = process_simulation(name_simulation, prefix, variables, f"{name_simulation}_*_sechiba_history.nc", solay_value=4.1 if "moistc" in variables else None)
    df_list.append(df)

# Concatenate all dataframes into one
final_df = pd.concat(df_list, axis=1)

# Reset index if necessary
final_df.reset_index(inplace=True)

# Display final dataframe
final_df = final_df.drop(columns=["time_centered", "lat", "lon"])


yes
yes2


KeyError: 'moistc'

In [66]:
# Function to process a single simulation
def process_simulation(name_simulation, prefix, variables, file_pattern, file_pattern_4dim=None, solay_value=None):
    # Open the 2D dataset
    output_nc = xr.open_mfdataset(f"{name_simulation}/SRF/Output/MO/{file_pattern}")
    # Extract the required variables and convert GPP columns
    data = output_nc[variables].to_dataframe()
    # Apply prefix to each variable
    data = data.rename(columns={var: f"{prefix}_{var}" for var in variables})
    # Apply GPP conversion (kgC/m²/s to gC/m²/day) to any column containing 'gpp'
    for col in data.columns:
        if "gpp" in col.lower():  # Check if the variable name contains 'gpp'
            data[col] = kgC_m2_s_to_gC_m2_day(data[col])
        elif "fluxsens" in col.lower() or "fluxlat" in col.lower():  # Check if the variable name contains 'fluxsens' or 'fluxlat'
            data[col] = w_m2_to_mj_m2_day(data[col])
    # If a 4D file is provided, handle moistc
    if file_pattern_4dim and solay_value is not None:
        # Open the 4D dataset and extract moistc
        output_4dim_nc = xr.open_mfdataset(f"{name_simulation}/SRF/Output/MO/{file_pattern_4dim}")
        moistc_data = output_4dim_nc.moistc.sel(soiltyp=2).sel(solay=solay_value, method="nearest").to_dataframe()
        moistc_data = moistc_data.rename(columns={"moistc": f"{prefix}_moistc"})
        # Reset indices before merging
        data.reset_index(inplace=True)
        data.drop(columns=['lat', 'lon', 'time_centered'], inplace=True) 
        moistc_data.reset_index(inplace=True)
        moistc_data.drop(columns=['soiltyp', 'solay', 'lat', 'lon', 'time_centered'], inplace=True)
        # Merge moistc data into the 2D data
        data = data.merge(moistc_data, on='time_counter', how="left")
    return data
# Define simulation details
simulations = {
    "EXP-CompRain.CompGW": "R_avgSWCC_avg",
    "EXP-CompRain.FfluxGW": "R_avgSWCC_var",
    "EXP-RainFflux.CompGW": "R_varSWCC_avg",
    "EXP-RainFflux.GWFflux": "R_varSWCC_var",
}
variables = ["rain", "gppTree", "gppCrop", "fluxlat", "fluxsens"]
# Process each simulation and combine into one dataframe
df_list = []
for name_simulation, prefix in simulations.items():
    # Adjust file patterns for the regular and 4D datasets
    df = process_simulation(
        name_simulation, 
        prefix, 
        variables, 
        f"{name_simulation}_*_sechiba_history.nc", 
        file_pattern_4dim=f"{name_simulation}_*_sechiba_history_4dim.nc",
        solay_value=5  # Adjust this based on the value you're interested in
    )
    df_list.append(df)

# Concatenate all dataframes into one
final_df = pd.concat(df_list, axis=1)

In [68]:
final_df.head()

,time_counter,R_avgSWCC_avg_rain,R_avgSWCC_avg_gppTree,R_avgSWCC_avg_gppCrop,R_avgSWCC_avg_fluxlat,R_avgSWCC_avg_fluxsens,R_avgSWCC_avg_moistc,time_counter,R_avgSWCC_var_rain,R_avgSWCC_var_gppTree,...,R_varSWCC_avg_fluxlat,R_varSWCC_avg_fluxsens,R_varSWCC_avg_moistc,time_counter,R_varSWCC_var_rain,R_varSWCC_var_gppTree,R_varSWCC_var_gppCrop,R_varSWCC_var_fluxlat,R_varSWCC_var_fluxsens,R_varSWCC_var_moistc
0,2019-01-01 12:00:00,0.0,2.199028,0.0,3.144030,3.433277,0.244344,2019-01-01 12:00:00,0.0,2.199028,...,3.144030,3.433277,0.244344,2019-01-01 12:00:00,0.0,2.199028,0.0,3.144030,3.433277,0.228726
1,2019-01-02 12:00:00,0.0,2.351439,0.0,3.246813,3.975858,0.259714,2019-01-02 12:00:00,0.0,2.351439,...,3.246813,3.975858,0.259714,2019-01-02 12:00:00,0.0,2.351439,0.0,3.246813,3.975858,0.227541
2,2019-01-03 12:00:00,0.0,2.136424,0.0,3.177708,4.677784,0.259128,2019-01-03 12:00:00,0.0,2.136424,...,3.177708,4.677784,0.259128,2019-01-03 12:00:00,0.0,2.136424,0.0,3.177708,4.677784,0.226402
3,2019-01-04 12:00:00,0.0,2.748015,0.0,3.678103,5.231138,0.258543,2019-01-04 12:00:00,0.0,2.748015,...,3.678103,5.231138,0.258543,2019-01-04 12:00:00,0.0,2.748015,0.0,3.678103,5.231138,0.225263
4,2019-01-05 12:00:00,0.0,3.020371,0.0,3.833693,4.346555,0.257958,2019-01-05 12:00:00,0.0,3.020371,...,3.833693,4.346555,0.257958,2019-01-05 12:00:00,0.0,3.020371,0.0,3.833693,4.346555,0.224123


In [69]:
final_df.to_clipboard()

In [14]:
#Stomate folder